### Import Data

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost.sklearn import XGBClassifier
import xgboost as xgb

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print(os.getcwd())
df = pd.read_csv("../dataset/fab.csv")

### Data Information

In [ ]:
df.info()

# dtypes: float64(590) -> Sensor, int64(1) -> Pass_Fail, str(1) -> SensorTime
df.dtypes 

df.describe()

df.head()
df.tail()

### Set Target / Feature

In [ ]:
target_col = "Pass_Fail"

feature_cols = [
    col for col in df.columns 
    if col not in ['Pass_Fail', 'SensorTime']
]

X = df[feature_cols]
y = df[target_col]

### Split Target / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
) 

print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

### Data Visualization

- 정상/불량 비율 확인

In [ ]:
train_df = X_train.copy()
train_df[target_col] = y_train

print(y_train.value_counts())

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

plt.figure(figsize=(5,5))
sns.countplot(x=target_col, data=train_df)
plt.title('정상 / 불량')
plt.show()

- 센서별 Pass / Fail 데이터 분포

In [ ]:
palette = {
    -1: "#4C72B0",   # Pass
     1: "#DD8452"    # Fail
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, sensor in enumerate(feature_cols[0:4]):
    sns.histplot(data=train_df,
                x=sensor,
                hue=target_col,
                bins=50,
                kde=True,
                palette=palette,
                alpha=0.45,
                edgecolor="black",
                linewidth=0.4,
                multiple="layer",
                ax=axes[i])
    sns.rugplot(data=train_df,
                x=sensor,
                hue=target_col,
                palette=palette,
                height=0.08,
                alpha=0.7,
                linewidth=1.2,
                ax=axes[i],
                legend=False)

    axes[i].set_title(f"{sensor} 데이터 분포")
    axes[i].set_xlabel("실제 측정값")
    axes[i].set_ylabel("데이터 수")

### Data Preprocessing

- 결측치 처리 

In [ ]:
# 결측값이 높은 feature 제거
missing_ratio = X_train.isnull().mean().sort_values(ascending=False) * 100
high_missing_features = missing_ratio[missing_ratio > 70 ].index.tolist()

X_train = X_train.drop(columns=high_missing_features)
X_test = X_test.drop(columns=high_missing_features)

# 나머지 결측치 데이터 중앙값으로 처리
median_values = X_train.median()

X_train = X_train.fillna(median_values)
X_test = X_test.fillna(median_values)

print(X_train.isna().sum().sum())
print(X_test.isna().sum().sum())

- 중복 데이터 처리

In [ ]:
duplicate_count = df.duplicated(subset=feature_cols + [target_col]).sum()
duplicate_count
# 중복 데이터 확인 결과, 전체 행 기준 중복 데이터는 존재하지 않았다.

- 이상치 처리

In [ ]:
# 현재 X_train에 남아있는 컬럼 기준으로 feature_cols 갱신
feature_cols = X_train.columns.tolist()

outliers_result = {}

for col in feature_cols:
    Q1 = np.nanquantile(X_train[col], 0.25)
    Q3 = np.nanquantile(X_train[col], 0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = X_train[
        (X_train[col] < lower_bound) |
        (X_train[col] > upper_bound)
    ]

    outliers_result[col] = len(outliers)

outliers_result = pd.Series(outliers_result).sort_values(ascending=False)
top_features = outliers_result.head(3).index

train_df = X_train.copy()
train_df[target_col] = y_train

for col in top_features:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=train_df, x=target_col, y=col)
    plt.title(f"{col} by Pass_Fail")
    plt.show()

### Zero Variance

In [ ]:
thresholder = VarianceThreshold(threshold=0)
_ = thresholder.fit_transform(X_train)

mask = ~thresholder.get_support()
cols_zero_var_drop = np.asarray(feature_cols)[mask].tolist()

print(f'** {len(cols_zero_var_drop)} Features to Drop by Low Variance')
print(f'{cols_zero_var_drop}')

X_train = X_train.drop(columns=cols_zero_var_drop)
X_test = X_test.drop(columns=cols_zero_var_drop)

feature_cols = X_train.columns.tolist()

### Remove Collinear Features

In [ ]:
# Remove Collinear Features

corr_matrix = X_train[feature_cols].corr(method="pearson").abs()

threshold = 0.90
drop_cols = set()

for i in range(len(corr_matrix.columns) - 1):
    for j in range(i + 1):
        col = corr_matrix.columns[i + 1]
        row = corr_matrix.index[j]
        val = corr_matrix.iloc[j, i + 1]

        if val >= threshold:
            print(col, "|", row, "|", round(val, 2))
            drop_cols.add(col)

drop_cols = list(drop_cols)
print(len(drop_cols), "개")

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

feature_cols = X_train.columns.tolist()

### Check Variable

In [ ]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

print("문자형 컬럼 개수:", len(cat_cols))
print(cat_cols)

### StandardScaler

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### LogisticRegression

In [ ]:
model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_scaled, y_train)

lr_y_pred = model_lr.predict(X_test_scaled)

cm = confusion_matrix(y_test, lr_y_pred)
plt.rcParams['figure.figsize'] = (5, 5)
sns.set_theme(style = 'dark', font_scale = 1.4)
sns.heatmap(cm, annot = True, annot_kws = {"size": 15})

print("훈련 세트 점수: {:.3f}".format(model_lr.score(X_train_scaled, y_train)))
print("테스트 세트 점수:{:.3f}".format(model_lr.score(X_test_scaled, y_test)))

print(confusion_matrix(y_test, lr_y_pred, labels=[-1,1]))
print(classification_report(y_test, lr_y_pred, labels=[-1,1]))

### RandomForestClassfier

In [ ]:
model_rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=5,
    min_samples_leaf=10,
    min_samples_split=20,
    n_estimators=200
)
model_rf.fit(X_train, y_train)

rf_y_pred = model_rf.predict(X_test)

cm = confusion_matrix(y_test, rf_y_pred)
plt.rcParams['figure.figsize'] = (5, 5)
sns.set_theme(style = 'dark', font_scale = 1.4)
sns.heatmap(cm, annot = True, annot_kws = {"size": 15})

print("훈련 세트 점수: {:.3f}".format(model_rf.score(X_train, y_train)))
print("테스트 세트 점수:{:.3f}".format(model_rf.score(X_test, y_test)))

print(confusion_matrix(y_test, rf_y_pred, labels=[-1,1]))
print(classification_report(y_test, rf_y_pred, labels=[-1,1]))

### XGBoost

In [ ]:
y_train_xgb = y_train.replace(-1,0) 
y_test_xgb = y_test.replace(-1,0)

model_xgb = XGBClassifier(random_state=1)
model_xgb.fit(X_train, y_train_xgb)

xgb_y_pred = model_xgb.predict(X_test)

cm = confusion_matrix(y_test_xgb, xgb_y_pred, labels=[0, 1])
plt.rcParams['figure.figsize'] = (5, 5)
sns.set_theme(style = 'dark', font_scale = 1.4)
sns.heatmap(cm, annot = True, annot_kws = {"size": 15})

print("훈련 세트 점수: {:.3f}".format(model_xgb.score(X_train, y_train_xgb)))
print("테스트 세트 점수: {:.3f}".format(model_xgb.score(X_test, y_test_xgb)))

print(confusion_matrix(y_test_xgb, xgb_y_pred, labels=[0, 1]))
print(classification_report(y_test_xgb, xgb_y_pred, labels=[0, 1]))

### Model Visualization

In [ ]:
models = ["LogisticRegression", "RandomForestClassfier", "XGBoost"]
accuracy_list, precision_list, recall_list, f1_score_list = [], [], [], []

# LogisticRegression
accuracy_list.append(accuracy_score(y_test, lr_y_pred))
precision_list.append(precision_score(y_test, lr_y_pred))
recall_list.append(recall_score(y_test, lr_y_pred))
f1_score_list.append(f1_score(y_test, lr_y_pred))

# RandomForestClassfier
accuracy_list.append(accuracy_score(y_test, rf_y_pred))
precision_list.append(precision_score(y_test, rf_y_pred))
recall_list.append(recall_score(y_test, rf_y_pred))
f1_score_list.append(f1_score(y_test, rf_y_pred))

# XGBoost
accuracy_list.append(accuracy_score(y_test, xgb_y_pred))
precision_list.append(precision_score(y_test, xgb_y_pred))
recall_list.append(recall_score(y_test, xgb_y_pred))
f1_score_list.append(f1_score(y_test, xgb_y_pred))

In [ ]:
colors = sns.color_palette('pastel')[0:len(models)]

fig, ax = plt.subplots(2, 2, figsize=(13, 10))

ax[0, 0].bar(models, accuracy_list, color=colors)
ax[0, 0].set_title("accuracy", fontsize=15)
for i, v in enumerate(accuracy_list):
    ax[0, 0].text(i, v + 0.001, str(round(v, 3)), ha='center', va='bottom', fontsize=12)
ax[0, 0].set_ylim(0.8, 1.0)  

ax[0, 1].bar(models, precision_list, color=colors)
ax[0, 1].set_title("precision").set_fontsize(15)
for i, v in enumerate(precision_list):
    ax[0, 1].text(i, v + 0.01, str(round(v, 3)), ha='center', va='bottom', fontsize=12)
ax[0, 1].set_ylim(0, max(precision_list) + 0.1)

ax[1, 0].bar(models,recall_list, color=colors)
ax[1, 0].set_title("recall").set_fontsize(15)
for i, v in enumerate(recall_list):
    ax[1, 0].text(i, v + 0.01, str(round(v, 3)), ha='center', va='bottom', fontsize=12)
ax[1, 0].set_ylim(0, max(recall_list) + 0.1)

ax[1, 1].bar(models, f1_score_list, color=colors)
ax[1, 1].set_title("f1_score").set_fontsize(15)
for i, v in enumerate(f1_score_list):
    ax[1, 1].text(i, v + 0.01, str(round(v, 3)), ha='center', va='bottom', fontsize=12)
ax[1, 1].set_ylim(0, max(f1_score_list) + 0.1)

for ax in ax.flat:
    ax.tick_params(axis='x', labelsize=11, rotation=45)  # x축 라벨 회전

plt.tight_layout()  # subplot 간의 간격 조정
plt.show()